# Module 03 — Image Segmentation Fundamentals

We cover the full progression from FCN to U-Net and the key loss functions
used for training segmentation models.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from metrics import iou_score, dice_score, pixel_accuracy, mean_iou
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

## 1. FCN — Fully Convolutional Network

FCN converts a classification network into a dense prediction network
by replacing FC layers with 1×1 convolutions.

In [ ]:
class FCN8s(nn.Module):
    """Simplified FCN-8s with VGG-style encoder."""
    def __init__(self, num_classes=21):
        super().__init__()
        # Encoder
        self.pool3_conv = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2, 2),  # /2
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2, 2),  # /4
        )
        self.pool4_conv = nn.Sequential(
            nn.Conv2d(128, 256, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2, 2),  # /8
        )
        self.pool5_conv = nn.Sequential(
            nn.Conv2d(256, 512, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2, 2),  # /16
        )
        # Classifier (1x1 convs replacing FC)
        self.score_fr  = nn.Conv2d(512, num_classes, 1)
        self.score_p4  = nn.Conv2d(256, num_classes, 1)
        self.score_p3  = nn.Conv2d(128, num_classes, 1)
        # Upsampling
        self.up2x  = nn.ConvTranspose2d(num_classes, num_classes, 4, stride=2, padding=1)
        self.up4x  = nn.ConvTranspose2d(num_classes, num_classes, 4, stride=2, padding=1)
        self.up8x  = nn.ConvTranspose2d(num_classes, num_classes, 16, stride=8, padding=4)

    def forward(self, x):
        p3 = self.pool3_conv(x)
        p4 = self.pool4_conv(p3)
        p5 = self.pool5_conv(p4)
        score = self.score_fr(p5)
        score = self.up2x(score) + self.score_p4(p4)
        score = self.up4x(score) + self.score_p3(p3)
        return self.up8x(score)

fcn = FCN8s(21).to(DEVICE)
x = torch.randn(1, 3, 256, 256).to(DEVICE)
y = fcn(x)
print('FCN output shape:', y.shape)  # (1, 21, 256, 256)

## 2. U-Net from Scratch

U-Net adds skip connections that concatenate encoder feature maps to the decoder.

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.net(x)

class UNet(nn.Module):
    def __init__(self, in_channels=3, num_classes=2, features=[64,128,256,512]):
        super().__init__()
        self.encoders = nn.ModuleList()
        self.decoders = nn.ModuleList()
        self.pool = nn.MaxPool2d(2)
        in_c = in_channels
        for f in features:
            self.encoders.append(DoubleConv(in_c, f))
            in_c = f
        self.bottleneck = DoubleConv(features[-1], features[-1]*2)
        for f in reversed(features):
            self.decoders.append(nn.ConvTranspose2d(f*2, f, 2, 2))
            self.decoders.append(DoubleConv(f*2, f))
        self.head = nn.Conv2d(features[0], num_classes, 1)

    def forward(self, x):
        skips = []
        for enc in self.encoders:
            x = enc(x); skips.append(x); x = self.pool(x)
        x = self.bottleneck(x)
        skips = skips[::-1]
        for i in range(0, len(self.decoders), 2):
            x = self.decoders[i](x)  # upsample
            s = skips[i//2]
            if x.shape != s.shape:
                x = F.interpolate(x, size=s.shape[2:], mode='bilinear', align_corners=False)
            x = torch.cat([s, x], dim=1)
            x = self.decoders[i+1](x)  # conv
        return self.head(x)

unet = UNet(3, 2).to(DEVICE)
x = torch.randn(1, 3, 256, 256).to(DEVICE)
y = unet(x)
print('U-Net output shape:', y.shape)  # (1, 2, 256, 256)
print('Parameters:', sum(p.numel() for p in unet.parameters())/1e6, 'M')

## 3. Segmentation Loss Functions

In [ ]:
class DiceLoss(nn.Module):
    def __init__(self, smooth=1e-6):
        super().__init__()
        self.smooth = smooth
    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        inter = (probs * targets).sum(dim=(1,2,3))
        total = probs.sum(dim=(1,2,3)) + targets.sum(dim=(1,2,3))
        return (1 - (2*inter + self.smooth) / (total + self.smooth)).mean()

class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha, self.gamma = alpha, gamma
    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(logits, targets.float(), reduction='none')
        pt = torch.exp(-bce)
        return (self.alpha * (1-pt)**self.gamma * bce).mean()

# Demo: compare losses on identical inputs
logits = torch.randn(2, 1, 64, 64)
targets = (torch.rand(2, 1, 64, 64) > 0.5).float()
bce  = F.binary_cross_entropy_with_logits(logits, targets)
dice = DiceLoss()(logits, targets)
focal= FocalLoss()(logits, targets)
print(f'BCE:   {bce.item():.4f}')
print(f'Dice:  {dice.item():.4f}')
print(f'Focal: {focal.item():.4f}')

## Exercise — Tversky Loss

The **Tversky Loss** generalises Dice by allowing separate weights `α` and `β`
for false positives and false negatives:

```
TI = TP / (TP + α·FP + β·FN)
Tversky Loss = 1 - TI
```

Dice is the special case where `α = β = 0.5`. Setting `β > α` penalises
false negatives more, which is useful when missing a positive is more costly
(e.g., tumour segmentation).

**Task:** implement `TverskyLoss(alpha, beta)`.

In [ ]:
### EXERCISE
class TverskyLoss(nn.Module):
    def __init__(self, alpha=0.3, beta=0.7, smooth=1e-6):
        super().__init__()
        # TODO: implement
        raise NotImplementedError
    def forward(self, logits, targets):
        raise NotImplementedError